# Repaso práctico · Clasificación y regresión

**20 ejercicios + 8 preguntas breves.** Nivel básico de repaso.

1. Desarrollo de un modelo.
2. KNN, random forest y naive Bayes.
3. OvO, OvR, GridSearchCV e imbalanced-learn.
4. De clasificación a regresión.
5. Regresión lineal, KNN, árbol y random forest.

Completa los `...`, añade lo que falta y corrige las celdas marcadas.
Responde las preguntas en una o dos frases. Ejecuta de arriba abajo **después de completar cada ejercicio**.
Las celdas de preparación se ejecutan tal cual. Los datos no necesitan descargas.

Borja Fernández Antelo · [borjafernandez.es](https://borjafernandez.es)

**Versión resuelta · Código completo y respuestas breves.**

## Preparación

Si falta alguna librería, descomenta y ejecuta esta celda. Después, reinicia el kernel.
`imbalanced-learn` se instala con ese nombre y se importa como `imblearn`.

In [1]:
# %pip install numpy pandas scikit-learn imbalanced-learn

In [2]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris, make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, recall_score, mean_absolute_error

## 1 · Pasos de desarrollo de un modelo

Caso: identificar una especie de flor a partir de cuatro medidas. Cada fila es una flor.

### Ejercicio 01 · Ordena el proceso

Ordena estas acciones. Escribe las letras en orden.

- A. Ajustar el preprocesado con entrenamiento y entrenar/ajustar modelos con validación cruzada.
- B. Definir qué predecir y cómo medir el éxito.
- C. Evaluar el modelo elegido con el test reservado.
- D. Recoger los datos y comprobar su calidad.
- E. Separar entrenamiento y test antes de aprender transformaciones.
- F. Poner el modelo en uso y vigilar su rendimiento.
- G. Explorar los datos de entrenamiento: distribuciones, nulos y clases.

**Respuesta:** B → D → E → G → A → C → F. Si la evaluación no cumple el objetivo, se revisa el proceso; no es una cadena de una sola pasada.

**Preparación · Iris**

In [3]:
iris = load_iris(as_frame=True)
flores = iris.frame
display(flores.head(3))

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0


### Ejercicio 02 · Separa entradas, objetivo y test

Completa `Xc` con todas las columnas salvo `target` y `yc` con el objetivo.
Reserva un 25 % para test. Conserva aproximadamente las proporciones de las clases.

In [4]:
Xc = flores.drop(columns='target')
yc = flores['target']
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    Xc, yc, test_size=0.25, random_state=42, stratify=yc
)
print(Xc_train.shape, Xc_test.shape)

(112, 4) (38, 4)


### Ejercicio 03 · Corrige el escalado

**Código con dos fallos.** El escalador aprende de todos los datos y vuelve a aprender del test.
Corrige las dos llamadas para que solo aprenda de entrenamiento.

In [5]:
escala = StandardScaler()
escala.fit(Xc_train)
Xc_train_s = escala.transform(Xc_train)
Xc_test_s = escala.transform(Xc_test)
print(Xc_train_s.shape, Xc_test_s.shape)

(112, 4) (38, 4)


### Pregunta 1

Un compañero entrena y evalúa con las mismas filas. Obtiene un 100 % de acierto. ¿Qué falta para saber si generaliza?

**Respuesta:** Falta evaluar con datos que no haya usado para entrenar ni ajustar decisiones. El acierto en entrenamiento no demuestra que funcione con ejemplos nuevos.

## 2 · KNN, random forest y naive Bayes

Usaremos el mismo reparto de Iris. Los parámetros de este bloque ya están fijados:
el test sirve para comprobarlos, no para ir cambiándolos hasta mejorar su resultado.

### Ejercicio 04 · Completa KNN y detecta qué falta

Usa 5 vecinos. Falta una línea entre la creación del modelo y la predicción: añádela.
Utiliza las entradas escaladas del ejercicio 3.

In [6]:
from sklearn.neighbors import KNeighborsClassifier
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(Xc_train_s, yc_train)
pred_knn = knn.predict(Xc_test_s)
print('Accuracy:', round(accuracy_score(yc_test, pred_knn), 3))

Accuracy: 0.921


### Pregunta 2

Con KNN y pesos uniformes, los tres vecinos más cercanos tienen clases A, A y B. ¿Qué predice? ¿Por qué importa escalar las variables?

**Respuesta:** Predice A por mayoría. Una variable con números mucho mayores puede dominar la distancia aunque no sea la más útil; el escalado reduce ese efecto.

### Ejercicio 05 · Completa random forest

Crea 80 árboles y limita su profundidad a 4. Entrena y predice.
Aquí usamos las entradas originales: los árboles no necesitan escalado por unidades.

In [7]:
from sklearn.ensemble import RandomForestClassifier
bosque = RandomForestClassifier(n_estimators=80, max_depth=4, random_state=42)
bosque.fit(Xc_train, yc_train)
pred_bosque = bosque.predict(Xc_test)
print('Accuracy:', round(accuracy_score(yc_test, pred_bosque), 3))

Accuracy: 0.947


### Pregunta 3

¿Por qué un random forest no suele producir muchos árboles idénticos? Menciona las dos fuentes habituales de aleatoriedad.

**Respuesta:** Cada árbol se entrena con una muestra aleatoria con reemplazo de las filas (bootstrap, por defecto). En cada división se considera un subconjunto aleatorio de variables, lo que añade diversidad.

### Ejercicio 06 · Completa naive Bayes

Usa `GaussianNB` para estas medidas numéricas. Completa el entrenamiento y la predicción.

In [8]:
from sklearn.naive_bayes import GaussianNB
bayes = GaussianNB()
bayes.fit(Xc_train, yc_train)
pred_bayes = bayes.predict(Xc_test)
print('Accuracy:', round(accuracy_score(yc_test, pred_bayes), 3))

Accuracy: 0.921


### Pregunta 4

¿Qué simplificación hace naive Bayes sobre las variables? ¿Qué añade la variante GaussianNB?

**Respuesta:** Supone que las variables son independientes entre sí dada la clase. GaussianNB modela cada variable mediante una distribución normal dentro de cada clase.

## 3 · OvO, OvR, GridSearchCV e imblearn

OvO enfrenta pares de clases. OvR enfrenta cada clase contra el resto.
GridSearchCV prueba combinaciones de hiperparámetros con validación cruzada.

### Ejercicio 07 · Cuenta los clasificadores

Hay cuatro clases: A, B, C y D.

- OvR: ___ clasificadores. Escribe uno de los enfrentamientos.
- OvO: ___ clasificadores. Escribe todos los pares, sin repetirlos.

**Respuesta:** OvR: 4; por ejemplo, A contra B+C+D. OvO: 6; A-B, A-C, A-D, B-C, B-D y C-D. Para C clases: OvR usa C modelos y OvO usa C·(C−1)/2.

### Ejercicio 08 · Aplica ambas estrategias

Completa con `OneVsRestClassifier` y `OneVsOneClassifier`.
GaussianNB ya admite varias clases; lo envolvemos solo para practicar las dos estrategias.

In [9]:
from sklearn.multiclass import OneVsRestClassifier, OneVsOneClassifier
ovr = OneVsRestClassifier(GaussianNB())
ovo = OneVsOneClassifier(GaussianNB())
ovr.fit(Xc_train, yc_train)
ovo.fit(Xc_train, yc_train)
print('Modelos OvR:', len(ovr.estimators_))
print('Modelos OvO:', len(ovo.estimators_))

Modelos OvR: 3
Modelos OvO: 3


### Ejercicio 09 · Completa GridSearchCV

Prueba `k = 3, 5, 7` y pesos `uniform` / `distance`, con 3 particiones.
La pipeline aprende el escalado dentro de cada partición. Ajusta la búsqueda solo con entrenamiento.

In [10]:
from sklearn.model_selection import GridSearchCV
pipe_knn = Pipeline([('escala', StandardScaler()), ('knn', KNeighborsClassifier())])
rejilla = {'knn__n_neighbors': [3, 5, 7], 'knn__weights': ['uniform', 'distance']}
busqueda = GridSearchCV(pipe_knn, rejilla, cv=3, scoring='accuracy')
busqueda.fit(Xc_train, yc_train)
print(busqueda.best_params_)
print('Accuracy media de validación:', round(busqueda.best_score_, 3))

{'knn__n_neighbors': 7, 'knn__weights': 'uniform'}
Accuracy media de validación: 0.964


### Pregunta 5

Hay 6 combinaciones y cv=3. ¿Cuántos entrenamientos se hacen durante la búsqueda? Con refit=True, ¿cuántos hay en total? ¿Interviene el test?

**Respuesta:** Se hacen 6 × 3 = 18 entrenamientos durante la validación. Con el reajuste final del mejor modelo sobre todo entrenamiento son 19. El test no interviene en la búsqueda.

### Ejercicio 10 · Detecta una métrica engañosa

La clase 1 es la minoritaria. Un modelo siempre predice 0.
Completa las dos métricas y observa si detecta algún caso de la clase 1.

In [11]:
reales = [0] * 95 + [1] * 5
pred_mayoria = [0] * 100
print('Accuracy:', accuracy_score(reales, pred_mayoria))
print('Recall clase 1:', recall_score(reales, pred_mayoria, zero_division=0))

Accuracy: 0.95
Recall clase 1: 0.0


**Preparación · Datos desbalanceados**

`SMOTE` crea ejemplos sintéticos de la clase minoritaria interpolando entre vecinos.
Este ejemplo contiene solo variables numéricas. El test conserva la distribución original.

In [12]:
from imblearn.over_sampling import SMOTE
from collections import Counter
Xi, yi = make_classification(
    n_samples=400, n_features=4, n_informative=3, n_redundant=0,
    weights=[0.9, 0.1], random_state=42
)
Xi_train, Xi_test, yi_train, yi_test = train_test_split(
    Xi, yi, test_size=0.25, stratify=yi, random_state=42
)
print('Entrenamiento:', Counter(yi_train.tolist()), '| Test:', Counter(yi_test.tolist()))

Entrenamiento: Counter({0: 271, 1: 29}) | Test: Counter({0: 91, 1: 9})


### Ejercicio 11 · Corrige dónde se aplica SMOTE

**Código con fallo.** Cambia los argumentos de `fit_resample`: se debe remuestrear solo entrenamiento.
Esta celda permite ver los recuentos; en el siguiente ejercicio usaremos una pipeline para validación cruzada.

In [13]:
smote = SMOTE(random_state=42)
Xi_bal, yi_bal = smote.fit_resample(Xi_train, yi_train)
print('Antes:', Counter(yi_train.tolist()))
print('Después:', Counter(yi_bal.tolist()))

Antes: Counter({0: 271, 1: 29})
Después: Counter({1: 271, 0: 271})


### Ejercicio 12 · Mete SMOTE dentro de la validación

Usa `imblearn.pipeline.Pipeline`: admite pasos de remuestreo.
Completa el paso SMOTE y los datos de entrenamiento **originales**, sin remuestrear.
Aquí se elige `k` por recall; mejorar esta métrica no garantiza mejorar todas las demás.

In [14]:
from imblearn.pipeline import Pipeline as ImbPipeline
pipe_smote = ImbPipeline([
    ('escala', StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ('knn', KNeighborsClassifier())
])
ajuste_smote = GridSearchCV(
    pipe_smote, {'knn__n_neighbors': [3, 5]}, cv=3, scoring='recall'
)
ajuste_smote.fit(Xi_train, yi_train);

**Comprobación · Test original, sin remuestrear**

In [15]:
pred_smote = ajuste_smote.predict(Xi_test)
print('Mejor k:', ajuste_smote.best_params_)
print('Accuracy:', round(accuracy_score(yi_test, pred_smote), 3))
print('Recall clase 1:', round(recall_score(yi_test, pred_smote), 3))

Mejor k: {'knn__n_neighbors': 5}
Accuracy: 0.91
Recall clase 1: 0.667


### Pregunta 6

¿Por qué sería un error usar Xi_bal e yi_bal del ejercicio 11 en GridSearchCV? ¿Qué evita la pipeline del ejercicio 12?

**Respuesta:** SMOTE habría usado también filas que después caerían en validación, filtrando información entre particiones. La pipeline ajusta el escalado y genera ejemplos solo en la parte de entrenamiento de cada partición; ni validación ni test se remuestrean.

## 4 · De clasificación a regresión

**Misma vivienda, dos preguntas:** «¿consumo bajo, medio o alto?» es clasificación;
«¿cuántos kWh consumirá?» es regresión. Cambia lo que quieres predecir.

| Aspecto | Clasificación | Regresión |
|---|---|---|
| Salida | Una categoría | Una cantidad numérica |
| Ejemplo | Bajo / medio / alto | 245,7 kWh |
| Evaluación habitual | Accuracy, precisión, recall, F1 | MAE, RMSE, R² |
| KNN con pesos uniformes | Voto de los vecinos | Media de sus valores |

**En común:** entradas `X`, objetivo `y`, separación de datos, `fit`, `predict`
y evaluación con datos nuevos. Un `Pipeline` sigue siendo útil.

### Ejercicio 13 · Decide la tarea

Escribe **C** (clasificación) o **R** (regresión) y señala qué sería `y`.

1. Predecir el precio de venta de un coche.
2. Decidir si un correo es spam.
3. Estimar los minutos de espera de un pedido.
4. Clasificar una valoración como 0=negativa, 1=neutra o 2=positiva.

**Respuesta:** 1. R; y = precio de venta. 2. C; y = spam/no spam. 3. R; y = minutos de espera. 4. C; y = categoría de valoración, aunque esté codificada con números.

### Ejercicio 14 · Una recta y un error

Un modelo predice consumo con la recta **ŷ = 50 + 20 × horas**.

- Para 4 horas: predicción = ___ kWh.
- Si el consumo real es 140 kWh: error absoluto = ___ kWh.
- Con errores absolutos de 10, 20 y 0 kWh: MAE = ___ kWh.

**Respuesta:** Predicción = 130 kWh. Error absoluto = |140 − 130| = 10 kWh. MAE = (10 + 20 + 0) / 3 = 10 kWh. La pendiente indica 20 kWh más por cada hora adicional en este modelo.

### Pregunta 7

¿Codificar las clases como 0, 1 y 2 convierte el problema en regresión? ¿La regresión logística se usa aquí como clasificador o como regresor?

**Respuesta:** No: importa el significado del objetivo, no que esté escrito con números. La regresión logística se usa como clasificador; estima probabilidades de clase.

## 5 · Implementa y compara regresores

Caso sintético: predecir consumo mensual en **kWh** a partir de superficie, horas de uso
y temperatura. Son viviendas independientes; usamos un reparto aleatorio.
Las fórmulas generan datos de práctica, no describen un consumo real.

**Preparación · Datos de consumo**

In [16]:
rng = np.random.default_rng(42)
consumo = pd.DataFrame({
    'm2': rng.uniform(40, 150, 240),
    'horas_uso': rng.uniform(1, 10, 240),
    'temperatura': rng.uniform(5, 30, 240)
})
consumo['kwh'] = (80 + 4 * consumo['m2'] + 35 * consumo['horas_uso']
                  - 3 * consumo['temperatura'] + rng.normal(0, 20, 240))
display(consumo.head(3).round(1))

,m2,horas_uso,temperatura,kwh
0,125.1,8.5,15.2,837.4
1,88.3,8.2,10.4,682.9
2,134.4,3.1,19.7,686.0


### Ejercicio 15 · Separa el problema de regresión

Extrae `kwh` como objetivo y elimínalo de las entradas. Reserva un 25 % para test.
No uses `stratify` para estos valores continuos. Conservaremos el test hasta el ejercicio 20.

In [17]:
Xr = consumo.drop(columns='kwh')
yr = consumo['kwh']
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    Xr, yr, test_size=0.25, random_state=42
)
print(Xr_train.shape, Xr_test.shape)

(180, 3) (60, 3)


**Patrón:** crear → entrenar con `fit` → predecir con `predict`.
En los siguientes cuatro ejercicios solo veremos tres predicciones sobre entrenamiento;
todavía no sirven para medir la generalización. Los hiperparámetros ya están fijados.

### Ejercicio 16 · Regresión lineal

Completa el entrenamiento y la predicción de las tres primeras filas de entrenamiento.
Este modelo aprende una suma de variables multiplicadas por coeficientes, más un término independiente.

In [18]:
from sklearn.linear_model import LinearRegression
lineal = LinearRegression()
lineal.fit(Xr_train, yr_train)
print(lineal.predict(Xr_train.iloc[:3]).round(1))

[692.1 581.9 774.6]


### Ejercicio 17 · KNN para regresión

Usa `KNeighborsRegressor` con 5 vecinos y añade el escalador que falta en el primer paso.
Con pesos uniformes, predice la media de los valores de los vecinos.

In [19]:
from sklearn.neighbors import KNeighborsRegressor
knn_r = Pipeline([
    ('escala', StandardScaler()),
    ('modelo', KNeighborsRegressor(n_neighbors=5))
])
knn_r.fit(Xr_train, yr_train)
print(knn_r.predict(Xr_train.iloc[:3]).round(1))

[682.7 585.8 770.7]


### Ejercicio 18 · Corrige el árbol

**Código con fallo.** Se ha elegido una clase para clasificación, pero queremos una cantidad continua.
Corrige el import y el constructor. Conserva la profundidad 4.
Con el criterio por defecto, cada hoja predice la media del objetivo de sus filas de entrenamiento.

In [20]:
from sklearn.tree import DecisionTreeRegressor
arbol_r = DecisionTreeRegressor(max_depth=4, random_state=42)
arbol_r.fit(Xr_train, yr_train)
print(arbol_r.predict(Xr_train.iloc[:3]).round(1))

[641.  541.  759.1]


### Ejercicio 19 · Random forest para regresión

Completa con `RandomForestRegressor`: 80 árboles y profundidad máxima 6.
Su predicción combina las de los árboles mediante una media.

In [21]:
from sklearn.ensemble import RandomForestRegressor
bosque_r = RandomForestRegressor(n_estimators=80, max_depth=6, random_state=42)
bosque_r.fit(Xr_train, yr_train)
print(bosque_r.predict(Xr_train.iloc[:3]).round(1))

[687.8 573.1 764.7]


**Preparación · Una referencia sencilla**

`DummyRegressor` predice siempre la media de entrenamiento. Intentaremos mejorar esa referencia.
Comparamos modelos con las mismas particiones de validación; el test sigue reservado.

In [22]:
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import cross_val_score, KFold
modelos = {'Media': DummyRegressor(), 'Lineal': lineal, 'KNN': knn_r,
           'Árbol': arbol_r, 'Random forest': bosque_r}
cv_r = KFold(n_splits=3, shuffle=True, random_state=42)

### Ejercicio 20 · Compara, elige y evalúa

Completa los datos usados en validación cruzada. La puntuación
`neg_mean_absolute_error` devuelve el MAE con signo negativo: cambia el signo de su media.
Gana el menor MAE de validación. Después predice el test con ese modelo y calcula su MAE.

In [23]:
filas = []
for nombre, modelo in modelos.items():
    scores = cross_val_score(modelo, Xr_train, yr_train, cv=cv_r,
                             scoring='neg_mean_absolute_error')
    mae_cv = -scores.mean()
    filas.append({'Modelo': nombre, 'MAE validación (kWh)': mae_cv})
comparacion = pd.DataFrame(filas).sort_values('MAE validación (kWh)')
display(comparacion.round(2))

,Modelo,MAE validación (kWh)
1,Lineal,15.34
4,Random forest,27.72
2,KNN,30.50
3,Árbol,43.49
0,Media,141.84


In [24]:
elegido = comparacion.iloc[0]['Modelo']
mejor_r = modelos[elegido]
mejor_r.fit(Xr_train, yr_train)
pred_r = mejor_r.predict(Xr_test)
mae_test = mean_absolute_error(yr_test, pred_r)
print('Elegido:', elegido)
print('MAE test (kWh):', round(mae_test, 2))

Elegido: Lineal
MAE test (kWh): 17.27


**Comprobación · Cinco predicciones finales**

In [25]:
detalle = pd.DataFrame({'Real': yr_test, 'Predicción': pred_r})
detalle['Error absoluto'] = abs(detalle['Real'] - detalle['Predicción'])
display(detalle.head(5).round(1))

,Real,Predicción,Error absoluto
24,849.4,853.5,4.2
6,702.9,748.6,45.7
93,659.0,639.6,19.4
109,686.0,685.1,0.9
104,744.6,778.3,33.7


### Pregunta 8

¿Cómo interpretas el MAE de test en kWh? En estos datos gana un modelo: ¿demuestra que siempre será el mejor para cualquier problema?

**Respuesta:** El MAE es la media del tamaño de los errores en kWh, sin importar su signo; no es un porcentaje ni un límite para cada error. El ganador solo lo es en esta comparación y estos datos: el consumo se ha generado con una relación lineal y ruido, lo que favorece al modelo lineal.

---

**Documentación de consulta · Revisión: 29/10/2026 (Europe/Madrid)**

- [Separación, escalado y fuga de datos](https://scikit-learn.org/stable/common_pitfalls.html).
- [KNN: clasificación y regresión](https://scikit-learn.org/stable/modules/neighbors.html).
- [Random forest](https://scikit-learn.org/stable/modules/ensemble.html#forest) y [naive Bayes](https://scikit-learn.org/stable/modules/naive_bayes.html).
- [OvO](https://scikit-learn.org/stable/modules/generated/sklearn.multiclass.OneVsOneClassifier.html), [OvR](https://scikit-learn.org/stable/modules/generated/sklearn.multiclass.OneVsRestClassifier.html) y [GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html).
- [Remuestreo y fuga de datos](https://imbalanced-learn.org/stable/common_pitfalls.html).

Verificado con Python 3.11, scikit-learn 1.9.1 e imbalanced-learn 0.14.2.
Fundamentos de repaso; consulta las APIs si reutilizas el notebook con otras versiones.